### PTB-XL ECG Preprocessing
#### CardioFusion-XAI

This notebook creates the reproducible ECG input pipeline used by the
CardioFusion-XAI ECG model.

Pipeline:

Raw PTB-XL ECG
    ↓
Load 12-lead signal
    ↓
Validate raw signal
    ↓
0.5 Hz high-pass filtering
    ↓
Resampling to target sampling frequency
    ↓
Strict length validation
    ↓
Normalization
    ↓
Float32 conversion
    ↓
Save `.npy`
    ↓
Build reproducible manifest

Important principles:
- PTB-XL official folds are preserved.
- Patient leakage is explicitly checked.
- No random train/validation/test split is created here.
- Validation/test data are never used to fit normalization statistics.
- Raw PTB-XL files are never modified.
- Unexpected signal lengths are rejected rather than silently padded/cropped.
- Failed records are recorded and are not silently discarded.

In [1]:
from pathlib import Path
import ast
import json
import math
import time
import warnings

import numpy as np
import pandas as pd
import wfdb

from scipy.signal import butter, sosfiltfilt, resample_poly

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = Path("../../").resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "ecg" / "ptbxl"
PROCESSED_ROOT = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "ecg" /
    "ptbxl"
)

ARTIFACT_ROOT = (
    PROJECT_ROOT /
    "artifacts" /
    "preprocessing" /
    "ptbxl"
)

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root     :", PROJECT_ROOT)
print("Raw PTB-XL       :", RAW_DIR)
print("Processed root   :", PROCESSED_ROOT)
print("Artifact root    :", ARTIFACT_ROOT)

Project root     : D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service
Raw PTB-XL       : D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\raw\ecg\ptbxl
Processed root   : D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl
Artifact root    : D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\preprocessing\ptbxl


In [3]:
# REPRODUCIBLE PREPROCESSING CONFIGURATION

RANDOM_SEED = 42

# Target ECG representation
TARGET_FS = 100
DURATION_SECONDS = 10
NUM_LEADS = 12
TARGET_LENGTH = TARGET_FS * DURATION_SECONDS

# Input waveform source
SOURCE_WAVEFORM = "filename_lr"

# Filtering
HIGH_PASS_ENABLED = True
HIGH_PASS_CUTOFF_HZ = 0.5
HIGH_PASS_ORDER = 3

# Normalization options:
#   "none"
#   "record_zscore"
#   "train_global_zscore"
NORMALIZATION_MODE = "record_zscore"

# Numerical stability
NORMALIZATION_EPSILON = 1e-8

# Safety
ALLOW_SKIPPED_RECORDS = False

# Set to an integer for a small test run.
# Use None for the complete dataset.
MAX_RECORDS = None

# Progress logging
PROGRESS_EVERY = 500

print("Configuration loaded.")
print(f"Target sampling rate : {TARGET_FS} Hz")
print(f"Target duration      : {DURATION_SECONDS} seconds")
print(f"Target length       : {TARGET_LENGTH} samples")
print(f"Normalization        : {NORMALIZATION_MODE}")

Configuration loaded.
Target sampling rate : 100 Hz
Target duration      : 10 seconds
Target length       : 1000 samples
Normalization        : record_zscore


In [4]:
VERSION_NAME = (
    f"{TARGET_FS}hz_"
    f"{NORMALIZATION_MODE}"
)

VERSION_DIR = PROCESSED_ROOT / VERSION_NAME

WAVEFORM_DIR = VERSION_DIR / "waveforms"
MANIFEST_DIR = VERSION_DIR / "manifests"
STATS_DIR = VERSION_DIR / "stats"
LOG_DIR = VERSION_DIR / "logs"

for directory in [
    VERSION_DIR,
    WAVEFORM_DIR,
    MANIFEST_DIR,
    STATS_DIR,
    LOG_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Preprocessing version:")
print(VERSION_NAME)

print("\nOutput directory:")
print(VERSION_DIR)

Preprocessing version:
100hz_record_zscore

Output directory:
D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore


In [5]:
DATABASE_FILE = RAW_DIR / "ptbxl_database.csv"
SCP_FILE = RAW_DIR / "scp_statements.csv"
RECORDS100_DIR = RAW_DIR / "records100"
RECORDS500_DIR = RAW_DIR / "records500"

print("ptbxl_database.csv :", DATABASE_FILE.exists())
print("scp_statements.csv :", SCP_FILE.exists())
print("records100         :", RECORDS100_DIR.exists())
print("records500         :", RECORDS500_DIR.exists())

required = [
    DATABASE_FILE,
    SCP_FILE,
]

missing = [
    path
    for path in required
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Required PTB-XL files are missing:\n" +
        "\n".join(str(x) for x in missing)
    )

ptbxl_database.csv : True
scp_statements.csv : True
records100         : True
records500         : True


In [6]:
ptbxl = pd.read_csv(
    DATABASE_FILE,
    index_col=0
)

scp = pd.read_csv(
    SCP_FILE,
    index_col=0
)

print("PTB-XL shape:", ptbxl.shape)
print("SCP shape    :", scp.shape)

display(
    ptbxl.head()
)

PTB-XL shape: (21837, 27)
SCP shape    : (71, 12)


,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,scp_codes,heart_axis,infarction_stadium1,infarction_stadium2,validated_by,second_opinion,initial_autogenerated_report,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
ecg_id,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",NaN,NaN,NaN,NaN,False,False,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,"{'NORM': 80.0, 'SBRAD': 0.0}",NaN,NaN,NaN,NaN,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,"{'NORM': 100.0, 'SR': 0.0}",NaN,NaN,NaN,NaN,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,"{'NORM': 100.0, 'SR': 0.0}",NaN,NaN,NaN,NaN,False,False,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,"{'NORM': 100.0, 'SR': 0.0}",NaN,NaN,NaN,NaN,False,False,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr


In [7]:
DIAGNOSTIC_LABELS = [
    "NORM",
    "MI",
    "STTC",
    "CD",
    "HYP",
]

RHYTHM_LABELS = [
    "SR",
    "AFIB",
    "AFLT",
    "STACH",
    "SBRAD",
    "SARRH",
    "PSVT",
    "BIGU",
    "PACE",
]

print("Diagnostic labels:")
print(DIAGNOSTIC_LABELS)

print("\nRhythm labels:")
print(RHYTHM_LABELS)

Diagnostic labels:
['NORM', 'MI', 'STTC', 'CD', 'HYP']

Rhythm labels:
['SR', 'AFIB', 'AFLT', 'STACH', 'SBRAD', 'SARRH', 'PSVT', 'BIGU', 'PACE']


In [8]:
def parse_scp_codes(value):
    """
    Convert PTB-XL scp_codes representation into a dictionary.
    """
    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    try:
        parsed = ast.literal_eval(value)

        if isinstance(parsed, dict):
            return parsed

    except (
        ValueError,
        SyntaxError,
        TypeError
    ):
        pass

    return {}


ptbxl["scp_codes_parsed"] = (
    ptbxl["scp_codes"]
    .apply(parse_scp_codes)
)

empty_scp = (
    ptbxl["scp_codes_parsed"]
    .apply(lambda x: len(x) == 0)
)

print(
    "Empty parsed SCP annotations:",
    int(empty_scp.sum())
)

if empty_scp.any():
    raise ValueError(
        "Some PTB-XL records have empty/unparseable SCP annotations."
    )

Empty parsed SCP annotations: 0


In [9]:
required_scp_columns = [
    "diagnostic",
    "diagnostic_class",
]

missing_scp_columns = [
    column
    for column in required_scp_columns
    if column not in scp.columns
]

if missing_scp_columns:
    raise KeyError(
        f"Missing SCP columns: {missing_scp_columns}"
    )

diagnostic_code_to_class = {}

for code, row in scp.iterrows():

    if pd.isna(row["diagnostic"]):
        continue

    if row["diagnostic"] != 1:
        continue

    diagnostic_class = row["diagnostic_class"]

    if diagnostic_class in DIAGNOSTIC_LABELS:
        diagnostic_code_to_class[code] = (
            diagnostic_class
        )

print(
    "Diagnostic SCP mappings:",
    len(diagnostic_code_to_class)
)

display(
    pd.Series(
        diagnostic_code_to_class,
        name="diagnostic_class"
    )
    .rename_axis("scp_code")
    .reset_index()
    .sort_values(
        ["diagnostic_class", "scp_code"]
    )
)

Diagnostic SCP mappings: 44


,scp_code,diagnostic_class
11,1AVB,CD
43,2AVB,CD
41,3AVB,CD
15,CLBBB,CD
14,CRBBB,CD
33,ILBBB,CD
10,IRBBB,CD
12,IVCD,CD
8,LAFB,CD
24,LPFB,CD


In [10]:
def get_diagnostic_targets(codes):

    targets = {
        label: 0
        for label in DIAGNOSTIC_LABELS
    }

    for code in codes.keys():

        diagnostic_class = (
            diagnostic_code_to_class.get(code)
        )

        if diagnostic_class is not None:
            targets[diagnostic_class] = 1

    return targets


diagnostic_target_dicts = (
    ptbxl["scp_codes_parsed"]
    .apply(get_diagnostic_targets)
)

diagnostic_target_df = pd.DataFrame(
    diagnostic_target_dicts.tolist(),
    index=ptbxl.index
)

for label in DIAGNOSTIC_LABELS:
    ptbxl[label] = (
        diagnostic_target_df[label]
        .astype(np.float32)
    )

display(
    ptbxl[
        ["scp_codes_parsed"] +
        DIAGNOSTIC_LABELS
    ].head()
)

,scp_codes_parsed,NORM,MI,STTC,CD,HYP
ecg_id,,,,,,
1,"{'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}",1.0,0.0,0.0,0.0,0.0
2,"{'NORM': 80.0, 'SBRAD': 0.0}",1.0,0.0,0.0,0.0,0.0
3,"{'NORM': 100.0, 'SR': 0.0}",1.0,0.0,0.0,0.0,0.0
4,"{'NORM': 100.0, 'SR': 0.0}",1.0,0.0,0.0,0.0,0.0
5,"{'NORM': 100.0, 'SR': 0.0}",1.0,0.0,0.0,0.0,0.0


In [11]:
def get_rhythm_targets(codes):

    targets = {
        label: 0
        for label in RHYTHM_LABELS
    }

    for code in codes.keys():

        if code in RHYTHM_LABELS:
            targets[code] = 1

    return targets


rhythm_target_dicts = (
    ptbxl["scp_codes_parsed"]
    .apply(get_rhythm_targets)
)

rhythm_target_df = pd.DataFrame(
    rhythm_target_dicts.tolist(),
    index=ptbxl.index
)

for label in RHYTHM_LABELS:
    ptbxl[f"RHYTHM_{label}"] = (
        rhythm_target_df[label]
        .astype(np.float32)
    )

display(
    ptbxl[
        [f"RHYTHM_{x}" for x in RHYTHM_LABELS]
    ].head()
)

,RHYTHM_SR,RHYTHM_AFIB,RHYTHM_AFLT,RHYTHM_STACH,RHYTHM_SBRAD,RHYTHM_SARRH,RHYTHM_PSVT,RHYTHM_BIGU,RHYTHM_PACE
ecg_id,,,,,,,,,
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
TRAIN_FOLDS = list(range(1, 9))
VAL_FOLDS = [9]
TEST_FOLDS = [10]

ptbxl["split"] = "unused"

ptbxl.loc[
    ptbxl["strat_fold"].isin(TRAIN_FOLDS),
    "split"
] = "train"

ptbxl.loc[
    ptbxl["strat_fold"].isin(VAL_FOLDS),
    "split"
] = "val"

ptbxl.loc[
    ptbxl["strat_fold"].isin(TEST_FOLDS),
    "split"
] = "test"

invalid_split = (
    ~ptbxl["split"].isin(
        ["train", "val", "test"]
    )
)

if invalid_split.any():
    raise ValueError(
        "Some records were not assigned to train/val/test."
    )

display(
    ptbxl["split"]
    .value_counts()
    .reindex(["train", "val", "test"])
    .rename("count")
    .to_frame()
)

,count
split,
train,17441
val,2193
test,2203


In [13]:
split_patients = {
    split: set(
        ptbxl.loc[
            ptbxl["split"] == split,
            "patient_id"
        ]
        .dropna()
        .astype(int)
    )
    for split in [
        "train",
        "val",
        "test"
    ]
}

train_val_overlap = (
    split_patients["train"] &
    split_patients["val"]
)

train_test_overlap = (
    split_patients["train"] &
    split_patients["test"]
)

val_test_overlap = (
    split_patients["val"] &
    split_patients["test"]
)

print(
    "Train ∩ Val :",
    len(train_val_overlap)
)

print(
    "Train ∩ Test:",
    len(train_test_overlap)
)

print(
    "Val ∩ Test  :",
    len(val_test_overlap)
)

if any([
    train_val_overlap,
    train_test_overlap,
    val_test_overlap,
]):
    raise RuntimeError(
        "Patient leakage detected across splits."
    )

print("\nPatient-level leakage check passed.")

Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

Patient-level leakage check passed.


In [14]:
fold_split_table = (
    ptbxl
    .groupby(["strat_fold", "split"])
    .size()
    .rename("ecg_count")
    .reset_index()
)

display(
    fold_split_table
)

,strat_fold,split,ecg_count
0,1,train,2177
1,2,train,2184
2,3,train,2194
3,4,train,2175
4,5,train,2176
5,6,train,2178
6,7,train,2178
7,8,train,2179
8,9,val,2193
9,10,test,2203


In [15]:
EXPECTED_LEADS = [
    "I",
    "II",
    "III",
    "AVR",
    "AVL",
    "AVF",
    "V1",
    "V2",
    "V3",
    "V4",
    "V5",
    "V6",
]

print("Expected leads:")
print(EXPECTED_LEADS)

if len(EXPECTED_LEADS) != NUM_LEADS:
    raise ValueError(
        "Expected lead count does not match NUM_LEADS."
    )

Expected leads:
['I', 'II', 'III', 'AVR', 'AVL', 'AVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']


In [16]:
if SOURCE_WAVEFORM not in ptbxl.columns:
    raise KeyError(
        f"Waveform column '{SOURCE_WAVEFORM}' "
        "does not exist in PTB-XL metadata."
    )

missing_waveform_references = (
    ptbxl[SOURCE_WAVEFORM]
    .isna()
    .sum()
)

print(
    "Waveform references missing:",
    int(missing_waveform_references)
)

if missing_waveform_references:
    raise ValueError(
        "Some records do not have waveform references."
    )

Waveform references missing: 0


In [17]:
def load_ecg(record_path):
    """
    Load one PTB-XL WFDB ECG.
    """
    signal, fields = wfdb.rdsamp(
        str(record_path)
    )

    signal = np.asarray(
        signal,
        dtype=np.float64
    )

    return signal, fields

In [18]:
def validate_raw_signal(
    signal,
    fields,
    expected_leads
):
    """
    Strict validation before transformation.

    PTB-XL WFDB may represent AVR/AVL/AVF using uppercase.
    We normalize lead names to uppercase for comparison only.
    The signal column order itself is not changed.
    """

    if signal.ndim != 2:
        raise ValueError(
            f"Expected 2D signal, got {signal.shape}"
        )

    if signal.shape[1] != len(expected_leads):
        raise ValueError(
            f"Expected {len(expected_leads)} leads, "
            f"got {signal.shape[1]}"
        )

    if not np.isfinite(signal).all():
        raise ValueError(
            "Signal contains NaN or Inf values."
        )

    if fields.get("n_sig") != len(expected_leads):
        raise ValueError(
            f"WFDB reports n_sig={fields.get('n_sig')} "
            f"but expected {len(expected_leads)}."
        )

    actual_leads = [
        str(x).strip().upper()
        for x in fields.get("sig_name", [])
    ]

    expected_leads_normalized = [
        str(x).strip().upper()
        for x in expected_leads
    ]

    if actual_leads != expected_leads_normalized:
        raise ValueError(
            "Lead order/names do not match expected PTB-XL order.\n"
            f"Expected: {expected_leads_normalized}\n"
            f"Actual:   {actual_leads}"
        )

    original_fs = float(fields["fs"])

    if not math.isfinite(original_fs):
        raise ValueError(
            "Invalid sampling frequency."
        )

    if original_fs <= 0:
        raise ValueError(
            f"Invalid sampling frequency: {original_fs}"
        )

    return original_fs

In [19]:
def highpass_filter(
    signal,
    fs,
    cutoff_hz=0.5,
    order=3
):
    """
    Zero-phase high-pass Butterworth filter.

    Operates along time axis.
    """

    if cutoff_hz <= 0:
        raise ValueError(
            "High-pass cutoff must be positive."
        )

    if cutoff_hz >= fs / 2:
        raise ValueError(
            "High-pass cutoff must be below Nyquist."
        )

    sos = butter(
        N=order,
        Wn=cutoff_hz,
        btype="highpass",
        fs=fs,
        output="sos",
    )

    filtered = sosfiltfilt(
        sos,
        signal,
        axis=0
    )

    if not np.isfinite(filtered).all():
        raise ValueError(
            "Filtering produced NaN/Inf values."
        )

    return filtered

In [20]:
def resample_signal(
    signal,
    original_fs,
    target_fs
):
    """
    Polyphase resampling.
    """

    original_fs_int = int(round(original_fs))
    target_fs_int = int(round(target_fs))

    if original_fs_int <= 0:
        raise ValueError(
            f"Invalid original sampling rate: {original_fs}"
        )

    if target_fs_int <= 0:
        raise ValueError(
            f"Invalid target sampling rate: {target_fs}"
        )

    if original_fs_int == target_fs_int:
        return signal.astype(
            np.float64,
            copy=False
        )

    gcd = np.gcd(
        original_fs_int,
        target_fs_int
    )

    up = target_fs_int // gcd
    down = original_fs_int // gcd

    resampled = resample_poly(
        signal,
        up=up,
        down=down,
        axis=0
    )

    if not np.isfinite(resampled).all():
        raise ValueError(
            "Resampling produced NaN/Inf values."
        )

    return resampled

In [21]:
def validate_target_length(
    signal,
    target_length
):
    """
    Reject unexpected signal lengths.

    No silent truncation.
    No silent zero-padding.
    """

    if signal.ndim != 2:
        raise ValueError(
            f"Expected [time, leads], got {signal.shape}"
        )

    actual_length = signal.shape[0]

    if actual_length != target_length:
        raise ValueError(
            f"Unexpected processed ECG length: "
            f"{actual_length} != {target_length}"
        )

    return signal

In [22]:
def normalize_record_zscore(
    signal,
    epsilon=1e-8
):
    """
    Normalize each ECG independently,
    separately for each lead.

    Mean/std are computed only from this ECG.
    """

    signal = signal.astype(
        np.float32,
        copy=False
    )

    mean = signal.mean(
        axis=0,
        keepdims=True
    )

    std = signal.std(
        axis=0,
        keepdims=True
    )

    std = np.maximum(
        std,
        epsilon
    )

    normalized = (
        signal - mean
    ) / std

    if not np.isfinite(normalized).all():
        raise ValueError(
            "Record normalization produced NaN/Inf."
        )

    return normalized.astype(
        np.float32
    )

In [23]:
def normalize_signal(
    signal,
    mode,
    train_mean=None,
    train_std=None,
    epsilon=1e-8
):
    """
    Apply the selected normalization strategy.
    """

    signal = signal.astype(
        np.float32,
        copy=False
    )

    if mode == "none":

        return signal

    if mode == "record_zscore":

        return normalize_record_zscore(
            signal,
            epsilon=epsilon
        )

    if mode == "train_global_zscore":

        if train_mean is None or train_std is None:
            raise ValueError(
                "Training normalization statistics "
                "are required for train_global_zscore."
            )

        train_mean = np.asarray(
            train_mean,
            dtype=np.float32
        ).reshape(1, -1)

        train_std = np.asarray(
            train_std,
            dtype=np.float32
        ).reshape(1, -1)

        train_std = np.maximum(
            train_std,
            epsilon
        )

        normalized = (
            signal - train_mean
        ) / train_std

        if not np.isfinite(normalized).all():
            raise ValueError(
                "Global normalization produced NaN/Inf."
            )

        return normalized.astype(
            np.float32
        )

    raise ValueError(
        f"Unknown normalization mode: {mode}"
    )

In [24]:
NORMALIZATION_MODE = "train_global_zscore"

In [25]:
def compute_train_global_stats(
    dataframe,
    raw_dir,
    waveform_column,
    target_fs,
    highpass_enabled,
    highpass_cutoff,
    highpass_order,
    target_length
):
    """
    Compute mean/std per lead using TRAIN records only.

    Streaming computation:
    sum(x)
    sum(x^2)
    count

    This avoids loading the entire training dataset into RAM.
    """

    train_df = dataframe[
        dataframe["split"] == "train"
    ]

    print(
        "Training records used for global statistics:",
        len(train_df)
    )

    running_sum = np.zeros(
        NUM_LEADS,
        dtype=np.float64
    )

    running_sum_sq = np.zeros(
        NUM_LEADS,
        dtype=np.float64
    )

    running_count = 0

    failures = []

    start_time = time.time()

    for i, (_, row) in enumerate(
        train_df.iterrows()
    ):

        record_name = row[waveform_column]
        record_path = raw_dir / record_name

        try:

            signal, fields = load_ecg(
                record_path
            )

            original_fs = validate_raw_signal(
                signal,
                fields,
                EXPECTED_LEADS
            )

            if highpass_enabled:

                signal = highpass_filter(
                    signal,
                    fs=original_fs,
                    cutoff_hz=highpass_cutoff,
                    order=highpass_order
                )

            signal = resample_signal(
                signal,
                original_fs=original_fs,
                target_fs=target_fs
            )

            signal = validate_target_length(
                signal,
                target_length=target_length
            )

            signal = signal.astype(
                np.float64,
                copy=False
            )

            running_sum += signal.sum(
                axis=0
            )

            running_sum_sq += np.square(
                signal
            ).sum(axis=0)

            running_count += signal.shape[0]

        except Exception as exc:

            failures.append({
                "ecg_id": row.name,
                "record_name": record_name,
                "error": str(exc),
            })

        if (i + 1) % PROGRESS_EVERY == 0:

            elapsed = time.time() - start_time

            print(
                f"Stats: {i + 1:,}/{len(train_df):,} "
                f"| elapsed={elapsed:.1f}s"
            )

    if failures:

        failure_df = pd.DataFrame(
            failures
        )

        failure_df.to_csv(
            LOG_DIR /
            "normalization_stats_failures.csv",
            index=False
        )

        raise RuntimeError(
            f"{len(failures)} training records failed "
            "while computing normalization statistics."
        )

    if running_count == 0:
        raise RuntimeError(
            "No samples were accumulated for normalization statistics."
        )

    mean = (
        running_sum /
        running_count
    )

    variance = (
        running_sum_sq /
        running_count
    ) - np.square(mean)

    variance = np.maximum(
        variance,
        0.0
    )

    std = np.sqrt(
        variance
    )

    std = np.maximum(
        std,
        NORMALIZATION_EPSILON
    )

    return (
        mean.astype(np.float32),
        std.astype(np.float32),
        running_count
    )

In [26]:
train_mean = None
train_std = None
train_stat_sample_count = None

if NORMALIZATION_MODE == "train_global_zscore":

    print(
        "Computing train-only normalization statistics..."
    )

    (
        train_mean,
        train_std,
        train_stat_sample_count
    ) = compute_train_global_stats(
        dataframe=ptbxl,
        raw_dir=RAW_DIR,
        waveform_column=SOURCE_WAVEFORM,
        target_fs=TARGET_FS,
        highpass_enabled=HIGH_PASS_ENABLED,
        highpass_cutoff=HIGH_PASS_CUTOFF_HZ,
        highpass_order=HIGH_PASS_ORDER,
        target_length=TARGET_LENGTH,
    )

    print("\nTraining mean per lead:")
    print(train_mean)

    print("\nTraining std per lead:")
    print(train_std)

    np.savez(
        STATS_DIR / "normalization_stats.npz",
        mean=train_mean,
        std=train_std,
        sample_count=train_stat_sample_count,
    )

    print(
        "\nSaved train-only normalization statistics."
    )

else:

    print(
        "Global training normalization not selected."
    )

Computing train-only normalization statistics...
Training records used for global statistics: 17441
Stats: 500/17,441 | elapsed=2.4s
Stats: 1,000/17,441 | elapsed=4.8s
Stats: 1,500/17,441 | elapsed=7.2s
Stats: 2,000/17,441 | elapsed=9.1s
Stats: 2,500/17,441 | elapsed=10.9s
Stats: 3,000/17,441 | elapsed=12.7s
Stats: 3,500/17,441 | elapsed=14.4s
Stats: 4,000/17,441 | elapsed=16.3s
Stats: 4,500/17,441 | elapsed=18.1s
Stats: 5,000/17,441 | elapsed=19.9s
Stats: 5,500/17,441 | elapsed=21.9s
Stats: 6,000/17,441 | elapsed=23.7s
Stats: 6,500/17,441 | elapsed=25.4s
Stats: 7,000/17,441 | elapsed=27.2s
Stats: 7,500/17,441 | elapsed=28.9s
Stats: 8,000/17,441 | elapsed=30.9s
Stats: 8,500/17,441 | elapsed=32.7s
Stats: 9,000/17,441 | elapsed=34.2s
Stats: 9,500/17,441 | elapsed=35.7s
Stats: 10,000/17,441 | elapsed=37.2s
Stats: 10,500/17,441 | elapsed=38.9s
Stats: 11,000/17,441 | elapsed=40.6s
Stats: 11,500/17,441 | elapsed=42.3s
Stats: 12,000/17,441 | elapsed=43.9s
Stats: 12,500/17,441 | elapsed=45.5s


In [27]:
def preprocess_record(
    record_path,
    target_fs,
    target_length,
    normalization_mode,
    highpass_enabled=True,
    highpass_cutoff=0.5,
    highpass_order=3,
    train_mean=None,
    train_std=None,
):
    """
    Complete deterministic preprocessing for one ECG.
    """

    signal, fields = load_ecg(
        record_path
    )

    original_shape = tuple(
        signal.shape
    )

    original_fs = validate_raw_signal(
        signal,
        fields,
        EXPECTED_LEADS
    )

    if highpass_enabled:

        signal = highpass_filter(
            signal,
            fs=original_fs,
            cutoff_hz=highpass_cutoff,
            order=highpass_order
        )

    filtered_shape = tuple(
        signal.shape
    )

    signal = resample_signal(
        signal,
        original_fs=original_fs,
        target_fs=target_fs
    )

    resampled_shape = tuple(
        signal.shape
    )

    signal = validate_target_length(
        signal,
        target_length
    )

    signal = normalize_signal(
        signal,
        mode=normalization_mode,
        train_mean=train_mean,
        train_std=train_std,
        epsilon=NORMALIZATION_EPSILON,
    )

    final_shape = tuple(
        signal.shape
    )

    if final_shape != (
        target_length,
        NUM_LEADS
    ):
        raise ValueError(
            f"Unexpected final shape: {final_shape}"
        )

    if signal.dtype != np.float32:
        signal = signal.astype(
            np.float32
        )

    if not np.isfinite(signal).all():
        raise ValueError(
            "Final signal contains NaN/Inf."
        )

    metadata = {
        "original_fs": float(original_fs),
        "original_shape": str(original_shape),
        "filtered_shape": str(filtered_shape),
        "resampled_shape": str(resampled_shape),
        "final_shape": str(final_shape),
        "final_dtype": str(signal.dtype),
    }

    return signal, metadata

In [28]:
sample_row = ptbxl.iloc[0]

sample_record_name = (
    sample_row[SOURCE_WAVEFORM]
)

sample_record_path = (
    RAW_DIR /
    sample_record_name
)

print("Sample record:")
print(sample_record_name)

processed_sample, sample_meta = (
    preprocess_record(
        record_path=sample_record_path,
        target_fs=TARGET_FS,
        target_length=TARGET_LENGTH,
        normalization_mode=NORMALIZATION_MODE,
        highpass_enabled=HIGH_PASS_ENABLED,
        highpass_cutoff=HIGH_PASS_CUTOFF_HZ,
        highpass_order=HIGH_PASS_ORDER,
        train_mean=train_mean,
        train_std=train_std,
    )
)

print("\nProcessed metadata:")
print(json.dumps(
    sample_meta,
    indent=2
))

print(
    "\nProcessed shape:",
    processed_sample.shape
)

print(
    "Processed dtype:",
    processed_sample.dtype
)

print(
    "Contains NaN:",
    np.isnan(processed_sample).any()
)

print(
    "Contains Inf:",
    np.isinf(processed_sample).any()
)

Sample record:
records100/00000/00001_lr

Processed metadata:
{
  "original_fs": 100.0,
  "original_shape": "(1000, 12)",
  "filtered_shape": "(1000, 12)",
  "resampled_shape": "(1000, 12)",
  "final_shape": "(1000, 12)",
  "final_dtype": "float32"
}

Processed shape: (1000, 12)
Processed dtype: float32
Contains NaN: False
Contains Inf: False


In [29]:
sample_mean = (
    processed_sample
    .mean(axis=0)
)

sample_std = (
    processed_sample
    .std(axis=0)
)

print("Per-lead means:")
print(
    np.round(
        sample_mean,
        5
    )
)

print("\nPer-lead std:")
print(
    np.round(
        sample_std,
        5
    )
)

if NORMALIZATION_MODE == "record_zscore":

    if not np.allclose(
        sample_mean,
        0.0,
        atol=1e-4
    ):
        raise AssertionError(
            "Record normalization mean check failed."
        )

    if not np.allclose(
        sample_std,
        1.0,
        atol=1e-4
    ):
        raise AssertionError(
            "Record normalization std check failed."
        )

    print(
        "\nRecord z-score validation passed."
    )

Per-lead means:
[ 0.00972  0.00993  0.00048 -0.01113  0.00526  0.00612 -0.00156 -0.00462
  0.00185  0.00037  0.00403  0.00697]

Per-lead std:
[0.649   0.52582 0.26591 0.65902 0.48585 0.31018 0.53823 0.66255 0.38444
 0.33351 0.36637 0.47285]


In [30]:
records_to_process = ptbxl.copy()

if MAX_RECORDS is not None:

    records_to_process = (
        records_to_process
        .iloc[:MAX_RECORDS]
        .copy()
    )

print(
    "Records selected:",
    len(records_to_process)
)

if MAX_RECORDS is None:
    print("Mode: FULL DATASET")
else:
    print(
        f"Mode: FIRST {MAX_RECORDS} RECORDS"
    )

Records selected: 21837
Mode: FULL DATASET


In [31]:
processed_rows = []
failed_rows = []

start_time = time.time()

total_records = len(
    records_to_process
)

print(
    f"Starting preprocessing of "
    f"{total_records:,} ECG records..."
)

for i, (ecg_id, row) in enumerate(
    records_to_process.iterrows()
):

    record_name = row[SOURCE_WAVEFORM]
    record_path = RAW_DIR / record_name

    # Use ECG ID as filename.
    # This prevents collisions caused by waveform basename.
    output_filename = (
        f"{int(ecg_id):08d}.npy"
    )

    output_path = (
        WAVEFORM_DIR /
        output_filename
    )

    try:

        signal, signal_meta = (
            preprocess_record(
                record_path=record_path,
                target_fs=TARGET_FS,
                target_length=TARGET_LENGTH,
                normalization_mode=NORMALIZATION_MODE,
                highpass_enabled=HIGH_PASS_ENABLED,
                highpass_cutoff=HIGH_PASS_CUTOFF_HZ,
                highpass_order=HIGH_PASS_ORDER,
                train_mean=train_mean,
                train_std=train_std,
            )
        )

        np.save(
            output_path,
            signal
        )

        processed_rows.append({
            "ecg_id": int(ecg_id),
            "patient_id": int(row["patient_id"]),
            "record_name": record_name,
            "processed_path": (
                output_path
                .relative_to(PROJECT_ROOT)
                .as_posix()
            ),
            "split": row["split"],
            "strat_fold": int(
                row["strat_fold"]
            ),
            "original_fs": signal_meta[
                "original_fs"
            ],
            "original_shape": signal_meta[
                "original_shape"
            ],
            "resampled_shape": signal_meta[
                "resampled_shape"
            ],
            "final_shape": signal_meta[
                "final_shape"
            ],
            "final_dtype": signal_meta[
                "final_dtype"
            ],
        })

    except Exception as exc:

        failed_rows.append({
            "ecg_id": int(ecg_id),
            "patient_id": int(row["patient_id"]),
            "record_name": record_name,
            "split": row["split"],
            "strat_fold": int(
                row["strat_fold"]
            ),
            "error_type": type(exc).__name__,
            "error": str(exc),
        })

    if (i + 1) % PROGRESS_EVERY == 0:

        elapsed = time.time() - start_time

        print(
            f"Processed {i + 1:,}/{total_records:,} "
            f"| success={len(processed_rows):,} "
            f"| failed={len(failed_rows):,} "
            f"| elapsed={elapsed:.1f}s"
        )

print("\nProcessing loop finished.")

print(
    "Successful:",
    len(processed_rows)
)

print(
    "Failed:",
    len(failed_rows)
)

Starting preprocessing of 21,837 ECG records...
Processed 500/21,837 | success=500 | failed=0 | elapsed=2.9s
Processed 1,000/21,837 | success=1,000 | failed=0 | elapsed=5.9s
Processed 1,500/21,837 | success=1,500 | failed=0 | elapsed=8.2s
Processed 2,000/21,837 | success=2,000 | failed=0 | elapsed=10.6s
Processed 2,500/21,837 | success=2,500 | failed=0 | elapsed=12.9s
Processed 3,000/21,837 | success=3,000 | failed=0 | elapsed=15.3s
Processed 3,500/21,837 | success=3,500 | failed=0 | elapsed=17.7s
Processed 4,000/21,837 | success=4,000 | failed=0 | elapsed=20.2s
Processed 4,500/21,837 | success=4,500 | failed=0 | elapsed=22.6s
Processed 5,000/21,837 | success=5,000 | failed=0 | elapsed=24.9s
Processed 5,500/21,837 | success=5,500 | failed=0 | elapsed=27.3s
Processed 6,000/21,837 | success=6,000 | failed=0 | elapsed=29.7s
Processed 6,500/21,837 | success=6,500 | failed=0 | elapsed=32.0s
Processed 7,000/21,837 | success=7,000 | failed=0 | elapsed=34.3s
Processed 7,500/21,837 | success=7,

In [32]:
failed_df = pd.DataFrame(
    failed_rows
)

failure_file = (
    LOG_DIR /
    "preprocessing_failures.csv"
)

failed_df.to_csv(
    failure_file,
    index=False
)

print(
    "Failure log saved:",
    failure_file
)

if len(failed_df) > 0:

    print("\nFailure summary:")

    display(
        failed_df[
            "error_type"
        ]
        .value_counts()
        .rename("count")
        .to_frame()
    )

    display(
        failed_df.head(20)
    )
else:

    print(
        "\nNo preprocessing failures."
    )

Failure log saved: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore\logs\preprocessing_failures.csv

No preprocessing failures.


In [33]:
if (
    len(failed_df) > 0
    and not ALLOW_SKIPPED_RECORDS
):

    raise RuntimeError(
        f"{len(failed_df)} ECG records failed preprocessing. "
        "See:\n"
        f"{failure_file}\n\n"
        "The pipeline has intentionally stopped instead of "
        "silently training on a reduced dataset."
    )

print(
    "Failure policy check passed."
)

Failure policy check passed.


In [34]:
processed_manifest = pd.DataFrame(
    processed_rows
)

print(
    "Processed manifest shape:",
    processed_manifest.shape
)

display(
    processed_manifest.head()
)

Processed manifest shape: (21837, 11)


,ecg_id,patient_id,record_name,processed_path,split,strat_fold,original_fs,original_shape,resampled_shape,final_shape,final_dtype
0,1,15709,records100/00000/00001_lr,data/processed/ecg/ptbxl/100hz_record_zscore/w...,train,3,100.0,"(1000, 12)","(1000, 12)","(1000, 12)",float32
1,2,13243,records100/00000/00002_lr,data/processed/ecg/ptbxl/100hz_record_zscore/w...,train,2,100.0,"(1000, 12)","(1000, 12)","(1000, 12)",float32
2,3,20372,records100/00000/00003_lr,data/processed/ecg/ptbxl/100hz_record_zscore/w...,train,5,100.0,"(1000, 12)","(1000, 12)","(1000, 12)",float32
3,4,17014,records100/00000/00004_lr,data/processed/ecg/ptbxl/100hz_record_zscore/w...,train,3,100.0,"(1000, 12)","(1000, 12)","(1000, 12)",float32
4,5,17448,records100/00000/00005_lr,data/processed/ecg/ptbxl/100hz_record_zscore/w...,train,4,100.0,"(1000, 12)","(1000, 12)","(1000, 12)",float32


In [35]:
label_columns = (
    DIAGNOSTIC_LABELS +
    [
        f"RHYTHM_{x}"
        for x in RHYTHM_LABELS
    ]
)

metadata_for_manifest = (
    ptbxl[
        [
            "patient_id",
            "strat_fold",
            "split"
        ] +
        label_columns
    ]
    .copy()
    .reset_index()
    .rename(
        columns={
            "index": "ecg_id"
        }
    )
)

metadata_for_manifest["ecg_id"] = (
    metadata_for_manifest["ecg_id"]
    .astype(int)
)

processed_manifest = (
    processed_manifest
    .merge(
        metadata_for_manifest,
        on=[
            "ecg_id",
            "patient_id",
            "strat_fold",
            "split"
        ],
        how="left",
        validate="one_to_one"
    )
)

print(
    "Manifest shape:",
    processed_manifest.shape
)

display(
    processed_manifest.head()
)

Manifest shape: (21837, 25)


,ecg_id,patient_id,record_name,processed_path,split,strat_fold,original_fs,original_shape,resampled_shape,final_shape,final_dtype,NORM,MI,STTC,CD,HYP,RHYTHM_SR,RHYTHM_AFIB,RHYTHM_AFLT,RHYTHM_STACH,RHYTHM_SBRAD,RHYTHM_SARRH,RHYTHM_PSVT,RHYTHM_BIGU,RHYTHM_PACE
0,1,15709,records100/00000/00001_lr,data/processed/ecg/ptbxl/100hz_record_zscore/w...,train,3,100.0,"(1000, 12)","(1000, 12)","(1000, 12)",float32,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,13243,records100/00000/00002_lr,data/processed/ecg/ptbxl/100hz_record_zscore/w...,train,2,100.0,"(1000, 12)","(1000, 12)","(1000, 12)",float32,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,3,20372,records100/00000/00003_lr,data/processed/ecg/ptbxl/100hz_record_zscore/w...,train,5,100.0,"(1000, 12)","(1000, 12)","(1000, 12)",float32,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,17014,records100/00000/00004_lr,data/processed/ecg/ptbxl/100hz_record_zscore/w...,train,3,100.0,"(1000, 12)","(1000, 12)","(1000, 12)",float32,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,17448,records100/00000/00005_lr,data/processed/ecg/ptbxl/100hz_record_zscore/w...,train,4,100.0,"(1000, 12)","(1000, 12)","(1000, 12)",float32,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [36]:
missing_manifest_labels = (
    processed_manifest[label_columns]
    .isna()
    .sum()
)

display(
    missing_manifest_labels
    .rename("missing")
    .to_frame()
)

if missing_manifest_labels.sum() > 0:
    raise AssertionError(
        "Manifest contains missing target labels."
    )

,missing
NORM,0
MI,0
STTC,0
CD,0
HYP,0
RHYTHM_SR,0
RHYTHM_AFIB,0
RHYTHM_AFLT,0
RHYTHM_STACH,0
RHYTHM_SBRAD,0


In [37]:
def resolve_processed_path(
    relative_path
):
    return (
        PROJECT_ROOT /
        Path(relative_path)
    )


processed_manifest["file_exists"] = (
    processed_manifest["processed_path"]
    .apply(
        lambda x:
        resolve_processed_path(x).exists()
    )
)

missing_files = (
    processed_manifest[
        ~processed_manifest["file_exists"]
    ]
)

print(
    "Missing processed files:",
    len(missing_files)
)

if len(missing_files) > 0:

    display(
        missing_files.head(20)
    )

    raise FileNotFoundError(
        "Manifest references missing processed files."
    )

processed_manifest = (
    processed_manifest
    .drop(columns=["file_exists"])
)

Missing processed files: 0


In [38]:
qc_rows = []

qc_start = time.time()

for i, row in processed_manifest.iterrows():

    array_path = (
        PROJECT_ROOT /
        row["processed_path"]
    )

    try:

        x = np.load(
            array_path,
            allow_pickle=False
        )

        qc_rows.append({
            "ecg_id": int(row["ecg_id"]),
            "shape": tuple(x.shape),
            "dtype": str(x.dtype),
            "nan_count": int(
                np.isnan(x).sum()
            ),
            "inf_count": int(
                np.isinf(x).sum()
            ),
            "all_zero": bool(
                np.all(x == 0)
            ),
            "finite": bool(
                np.isfinite(x).all()
            ),
            "min": float(
                np.min(x)
            ),
            "max": float(
                np.max(x)
            ),
            "mean": float(
                np.mean(x)
            ),
            "std": float(
                np.std(x)
            ),
        })

    except Exception as exc:

        qc_rows.append({
            "ecg_id": int(row["ecg_id"]),
            "shape": None,
            "dtype": None,
            "nan_count": None,
            "inf_count": None,
            "all_zero": None,
            "finite": False,
            "min": None,
            "max": None,
            "mean": None,
            "std": None,
            "error": str(exc),
        })

    if (
        (i + 1) % PROGRESS_EVERY == 0
    ):
        print(
            f"QC: {i + 1:,}/"
            f"{len(processed_manifest):,}"
        )

qc_df = pd.DataFrame(
    qc_rows
)

print(
    "\nQC finished in",
    round(time.time() - qc_start, 2),
    "seconds"
)

display(
    qc_df.head()
)

QC: 500/21,837
QC: 1,000/21,837
QC: 1,500/21,837
QC: 2,000/21,837
QC: 2,500/21,837
QC: 3,000/21,837
QC: 3,500/21,837
QC: 4,000/21,837
QC: 4,500/21,837
QC: 5,000/21,837
QC: 5,500/21,837
QC: 6,000/21,837
QC: 6,500/21,837
QC: 7,000/21,837
QC: 7,500/21,837
QC: 8,000/21,837
QC: 8,500/21,837
QC: 9,000/21,837
QC: 9,500/21,837
QC: 10,000/21,837
QC: 10,500/21,837
QC: 11,000/21,837
QC: 11,500/21,837
QC: 12,000/21,837
QC: 12,500/21,837
QC: 13,000/21,837
QC: 13,500/21,837
QC: 14,000/21,837
QC: 14,500/21,837
QC: 15,000/21,837
QC: 15,500/21,837
QC: 16,000/21,837
QC: 16,500/21,837
QC: 17,000/21,837
QC: 17,500/21,837
QC: 18,000/21,837
QC: 18,500/21,837
QC: 19,000/21,837
QC: 19,500/21,837
QC: 20,000/21,837
QC: 20,500/21,837
QC: 21,000/21,837
QC: 21,500/21,837

QC finished in 11.5 seconds


,ecg_id,shape,dtype,nan_count,inf_count,all_zero,finite,min,max,mean,std
0,1,"(1000, 12)",float32,0,0,False,True,-4.221621,4.182054,0.002285,0.489974
1,2,"(1000, 12)",float32,0,0,False,True,-9.898540,9.479263,-0.003414,1.041862
2,3,"(1000, 12)",float32,0,0,False,True,-5.637677,5.540639,0.003929,0.571439
3,4,"(1000, 12)",float32,0,0,False,True,-7.904612,10.345549,-0.008280,1.421877
4,5,"(1000, 12)",float32,0,0,False,True,-6.235535,8.269565,0.000103,0.892061


In [39]:
expected_shape = (
    TARGET_LENGTH,
    NUM_LEADS
)

bad_shape = (
    qc_df["shape"] != expected_shape
)

bad_nan = (
    qc_df["nan_count"] > 0
)

bad_inf = (
    qc_df["inf_count"] > 0
)

bad_zero = (
    qc_df["all_zero"] == True
)

bad_finite = (
    qc_df["finite"] != True
)

print(
    "Bad shapes   :",
    int(bad_shape.sum())
)

print(
    "NaN arrays   :",
    int(bad_nan.sum())
)

print(
    "Inf arrays   :",
    int(bad_inf.sum())
)

print(
    "All-zero     :",
    int(bad_zero.sum())
)

print(
    "Non-finite   :",
    int(bad_finite.sum())
)

if (
    bad_shape.any()
    or bad_nan.any()
    or bad_inf.any()
    or bad_zero.any()
    or bad_finite.any()
):
    raise RuntimeError(
        "Processed waveform QC failed."
    )

print(
    "\nFull processed-waveform QC passed."
)

Bad shapes   : 0
NaN arrays   : 0
Inf arrays   : 0
All-zero     : 0
Non-finite   : 0

Full processed-waveform QC passed.


In [40]:
if NORMALIZATION_MODE == "record_zscore":

    mean_error = np.abs(
        qc_df["mean"].to_numpy()
    )

    std_error = np.abs(
        qc_df["std"].to_numpy() - 1.0
    )

    print(
        "Maximum absolute global mean:",
        mean_error.max()
    )

    print(
        "Maximum deviation from unit global std:",
        std_error.max()
    )

    print(
        "\nNote:"
    )

    print(
        "The QC mean/std values above are computed over "
        "all 12 leads and all time samples in each ECG."
    )

    print(
        "They are not the per-lead statistics used during "
        "record normalization."
    )

In [41]:
processed_split_summary = (
    processed_manifest[
        "split"
    ]
    .value_counts()
    .reindex(
        ["train", "val", "test"]
    )
    .rename("processed_ecgs")
    .to_frame()
)

processed_split_summary["percentage"] = (
    processed_split_summary["processed_ecgs"]
    / len(processed_manifest)
    * 100
).round(2)

display(
    processed_split_summary
)

,processed_ecgs,percentage
split,,
train,17441,79.87
val,2193,10.04
test,2203,10.09


In [42]:
raw_split_counts = (
    ptbxl[
        "split"
    ]
    .value_counts()
    .reindex(
        ["train", "val", "test"]
    )
    .rename("raw_ecgs")
)

processed_split_counts = (
    processed_manifest[
        "split"
    ]
    .value_counts()
    .reindex(
        ["train", "val", "test"]
    )
    .rename("processed_ecgs")
)

split_comparison = pd.concat(
    [
        raw_split_counts,
        processed_split_counts
    ],
    axis=1
)

split_comparison["difference"] = (
    split_comparison["raw_ecgs"]
    -
    split_comparison["processed_ecgs"]
)

display(
    split_comparison
)

if (
    split_comparison["difference"]
    != 0
).any() and not ALLOW_SKIPPED_RECORDS:

    raise AssertionError(
        "Raw and processed split counts do not match."
    )

,raw_ecgs,processed_ecgs,difference
split,,,
train,17441,17441,0
val,2193,2193,0
test,2203,2203,0


In [43]:
raw_label_counts = {}

for label in DIAGNOSTIC_LABELS:

    raw_label_counts[label] = int(
        ptbxl[label].sum()
    )

for label in RHYTHM_LABELS:

    raw_label_counts[
        f"RHYTHM_{label}"
    ] = int(
        ptbxl[
            f"RHYTHM_{label}"
        ].sum()
    )


processed_label_counts = {
    column: int(
        processed_manifest[column].sum()
    )
    for column in label_columns
}

label_comparison = pd.DataFrame({
    "raw_count": pd.Series(
        raw_label_counts
    ),
    "processed_count": pd.Series(
        processed_label_counts
    ),
})

label_comparison["difference"] = (
    label_comparison["raw_count"]
    -
    label_comparison["processed_count"]
)

display(
    label_comparison
)

if (
    label_comparison["difference"]
    != 0
).any() and not ALLOW_SKIPPED_RECORDS:

    raise AssertionError(
        "Target label counts changed during preprocessing."
    )

,raw_count,processed_count,difference
NORM,9528,9528,0
MI,5486,5486,0
STTC,5250,5250,0
CD,4907,4907,0
HYP,2655,2655,0
RHYTHM_SR,16782,16782,0
RHYTHM_AFIB,1514,1514,0
RHYTHM_AFLT,73,73,0
RHYTHM_STACH,826,826,0
RHYTHM_SBRAD,637,637,0


In [44]:
manifest_path = (
    MANIFEST_DIR /
    "ptbxl_manifest.csv"
)

processed_manifest.to_csv(
    manifest_path,
    index=False
)

print(
    "Saved manifest:",
    manifest_path
)

Saved manifest: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore\manifests\ptbxl_manifest.csv


In [45]:
qc_path = (
    MANIFEST_DIR /
    "ptbxl_waveform_qc.csv"
)

qc_df.to_csv(
    qc_path,
    index=False
)

print(
    "Saved waveform QC:",
    qc_path
)

Saved waveform QC: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore\manifests\ptbxl_waveform_qc.csv


In [46]:
preprocessing_config = {
    "dataset": "PTB-XL",

    "version_name": VERSION_NAME,

    "random_seed": RANDOM_SEED,

    "source_waveform": SOURCE_WAVEFORM,

    "sampling": {
        "target_fs_hz": TARGET_FS,
        "duration_seconds": DURATION_SECONDS,
        "target_length": TARGET_LENGTH,
    },

    "signal": {
        "num_leads": NUM_LEADS,
        "lead_names": EXPECTED_LEADS,
    },

    "filter": {
        "enabled": HIGH_PASS_ENABLED,
        "type": "butterworth_highpass",
        "cutoff_hz": HIGH_PASS_CUTOFF_HZ,
        "order": HIGH_PASS_ORDER,
        "zero_phase": True,
    },

    "resampling": {
        "method": "scipy.signal.resample_poly",
    },

    "length_policy": {
        "mode": "strict",
        "truncate": False,
        "zero_pad": False,
    },

    "normalization": {
        "mode": NORMALIZATION_MODE,
        "epsilon": NORMALIZATION_EPSILON,
        "statistics_source": (
            "train_only"
            if NORMALIZATION_MODE ==
            "train_global_zscore"
            else "record"
            if NORMALIZATION_MODE ==
            "record_zscore"
            else "none"
        ),
    },

    "split": {
        "train_folds": TRAIN_FOLDS,
        "validation_folds": VAL_FOLDS,
        "test_folds": TEST_FOLDS,
    },

    "targets": {
        "diagnostic_labels": DIAGNOSTIC_LABELS,
        "rhythm_labels": RHYTHM_LABELS,
        "multilabel": True,
    },

    "safety": {
        "allow_skipped_records": ALLOW_SKIPPED_RECORDS,
    },
}

config_path = (
    VERSION_DIR /
    "preprocessing_config.json"
)

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        preprocessing_config,
        f,
        indent=2
    )

print(
    "Saved configuration:",
    config_path
)

Saved configuration: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore\preprocessing_config.json


In [47]:
normalization_metadata = {
    "mode": NORMALIZATION_MODE,
}

if NORMALIZATION_MODE == "train_global_zscore":

    normalization_metadata.update({
        "mean": train_mean.tolist(),
        "std": train_std.tolist(),
        "sample_count": int(
            train_stat_sample_count
        ),
        "fit_split": "train",
        "fit_folds": TRAIN_FOLDS,
    })

normalization_metadata_path = (
    STATS_DIR /
    "normalization_metadata.json"
)

with open(
    normalization_metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        normalization_metadata,
        f,
        indent=2
    )

print(
    "Saved:",
    normalization_metadata_path
)

Saved: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore\stats\normalization_metadata.json


In [48]:
summary = {
    "dataset": "PTB-XL",
    "total_raw_records": int(len(ptbxl)),
    "total_processed_records": int(
        len(processed_manifest)
    ),
    "failed_records": int(
        len(failed_df)
    ),
    "target_fs_hz": int(TARGET_FS),
    "duration_seconds": int(
        DURATION_SECONDS
    ),
    "target_length": int(
        TARGET_LENGTH
    ),
    "num_leads": int(NUM_LEADS),
    "normalization": NORMALIZATION_MODE,
    "train_records": int(
        (processed_manifest["split"] == "train").sum()
    ),
    "validation_records": int(
        (processed_manifest["split"] == "val").sum()
    ),
    "test_records": int(
        (processed_manifest["split"] == "test").sum()
    ),
    "patient_train_val_overlap": int(
        len(train_val_overlap)
    ),
    "patient_train_test_overlap": int(
        len(train_test_overlap)
    ),
    "patient_val_test_overlap": int(
        len(val_test_overlap)
    ),
}

summary_path = (
    VERSION_DIR /
    "preprocessing_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )

print(
    json.dumps(
        summary,
        indent=2
    )
)

{
  "dataset": "PTB-XL",
  "total_raw_records": 21837,
  "total_processed_records": 21837,
  "failed_records": 0,
  "target_fs_hz": 100,
  "duration_seconds": 10,
  "target_length": 1000,
  "num_leads": 12,
  "normalization": "train_global_zscore",
  "train_records": 17441,
  "validation_records": 2193,
  "test_records": 2203,
  "patient_train_val_overlap": 0,
  "patient_train_test_overlap": 0,
  "patient_val_test_overlap": 0
}


In [49]:
print("=" * 80)
print("FINAL PREPROCESSING INTEGRITY GATE")
print("=" * 80)

assert len(processed_manifest) > 0

assert len(failed_df) == 0

assert len(train_val_overlap) == 0
assert len(train_test_overlap) == 0
assert len(val_test_overlap) == 0

assert (
    processed_manifest["split"]
    .isin(["train", "val", "test"])
    .all()
)

assert (
    processed_manifest[label_columns]
    .notna()
    .all()
    .all()
)

assert (
    qc_df["finite"]
    .all()
)

assert (
    (qc_df["nan_count"] == 0)
    .all()
)

assert (
    (qc_df["inf_count"] == 0)
    .all()
)

assert (
    (qc_df["shape"] == expected_shape)
    .all()
)

print("[PASS] No preprocessing failures")
print("[PASS] No patient leakage")
print("[PASS] All target labels present")
print("[PASS] All processed files exist")
print("[PASS] All processed arrays finite")
print("[PASS] No NaN values")
print("[PASS] No Inf values")
print("[PASS] All arrays have expected shape")
print("[PASS] Split integrity preserved")

print("\nPreprocessing integrity gate PASSED.")

FINAL PREPROCESSING INTEGRITY GATE
[PASS] No preprocessing failures
[PASS] No patient leakage
[PASS] All target labels present
[PASS] All processed files exist
[PASS] All processed arrays finite
[PASS] No NaN values
[PASS] No Inf values
[PASS] All arrays have expected shape
[PASS] Split integrity preserved

Preprocessing integrity gate PASSED.


In [50]:
print("=" * 80)
print("PTB-XL PREPROCESSING COMPLETE")
print("=" * 80)

print(
    f"Version          : {VERSION_NAME}"
)

print(
    f"Processed ECGs   : {len(processed_manifest):,}"
)

print(
    f"Sampling rate    : {TARGET_FS} Hz"
)

print(
    f"Duration         : {DURATION_SECONDS} seconds"
)

print(
    f"Input shape      : ({TARGET_LENGTH}, {NUM_LEADS})"
)

print(
    f"Normalization     : {NORMALIZATION_MODE}"
)

print(
    f"Train             : "
    f"{(processed_manifest['split'] == 'train').sum():,}"
)

print(
    f"Validation        : "
    f"{(processed_manifest['split'] == 'val').sum():,}"
)

print(
    f"Test              : "
    f"{(processed_manifest['split'] == 'test').sum():,}"
)

print(
    "\nWaveforms:"
)

print(
    WAVEFORM_DIR
)

print(
    "\nManifest:"
)

print(
    manifest_path
)

print(
    "\nConfiguration:"
)

print(
    config_path
)

print(
    "\nQC:"
)

print(
    qc_path
)

PTB-XL PREPROCESSING COMPLETE
Version          : 100hz_record_zscore
Processed ECGs   : 21,837
Sampling rate    : 100 Hz
Duration         : 10 seconds
Input shape      : (1000, 12)
Normalization     : train_global_zscore
Train             : 17,441
Validation        : 2,193
Test              : 2,203

Waveforms:
D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore\waveforms

Manifest:
D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore\manifests\ptbxl_manifest.csv

Configuration:
D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore\preprocessing_config.json

QC:
D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\data\processed\ecg\ptbxl\100hz_record_zscore\manifests\ptbxl_waveform_qc.csv
